# julia_macro (v2, stage 4) — `ggb"…"g`: the construction text goes to the Python closed-world parser through PythonCall (ruling A, 2026-09-11)

v1's `@ggb` was an Expr macro: Julia parsed the GeoGebra text first, so prime labels (`C'`), juxtaposition (`u v`), numeric literals (`0.6260`) and operator printing were rewritten or rejected before the macro ran. v2: a non-standard string literal — Julia never parses the body — and ONE parser (`ggblab.parse`, 28 heads) for both kernels.

In [1]:
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"; ENV["JULIA_PYTHONCALL_EXE"] = "/Users/manabu/miniforge3/envs/py314/bin/python3"   # the lab's python (ggblab on PYTHONPATH)
include("/Users/manabu/work/ggblab-replay/julia/host/html_host.jl"); include("/Users/manabu/work/ggblab-replay/julia/host/ggb_macro.jl"); using .GGBLabHost, .GGBLabMacro, PythonCall
GGBLabHost.DEPLOY[] = "https://cdn.geogebra.org/apps/deployggb.js"
g = GeoGebra(appName="suite", showAlgebraInput=true)
println("kernel ", g.kernel_id, " | box ", g.mount_id, " | python ", PythonCall.C.CTX.exe_path)
g

## the eg11 construction, with what v1's Expr macro could not carry: prime labels, `2π/5`, a trailing-zero literal, `x²`

In [2]:
t0 = time()
lbls = ggb"""
:const :new
A = (0, 0)
c = Circle(:A, 1)
B = (cos(2π/5), sin(2π/5))
l = Line(:A, :B)
C = Point(:l)
p = PerpendicularLine(:C, :l)
lst1 = {Intersect(c, p)}
C' = lst1(1)
A' = lst1(2)
t1 = Polygon(:A, :C', :A')
r = 0.6260
d = Distance(:A, :C)² + r
nL1 = Length(lst1)
"""g
println("LABELS ", lbls, " in ", round(time() - t0; digits=2), " s")
println("errors: ", errors(g))

In [3]:
c = ggb"""
C' = lst1(1)
B = (cos(2π/5), sin(2π/5))
r = 0.6260
d = Distance(:A, :C)² + r
"""
println("to_ggb (byte for byte): ", to_ggb(c))
println("d = ", value(g, "d"), " | r = ", value(g, "r"), " | nL1 = ", value(g, "nL1"))

## the closed world crosses the boundary (UnknownHead / RelativeReference are Python's; here they are `ClosedWorldError`)

In [4]:
for s in ("Cylinder(A, B, 1)", "Circle(_1, 1)")
    try
        parse_ggb(s); println("NO-ERROR")
    catch e
        println(first(sprint(showerror, e), 110))
    end
end

## read back through the Python data-in (C6: `ggblab_extra.geometry_ir` is not reimplemented in Julia)

In [5]:
x = xml(g; timeout=60.0)
ir = pyimport("ggblab_extra.geometry_ir")
irs = ir.element_irs(x)
for lab in ("C'", "A'", "B")
    e = irs[lab]; println(lab, " ", pyconvert(String, e.type), " ", pyconvert(Any, e.coords))
end
println("DONE")